# RAG Pipeline Exercise — SOLUTION

This notebook contains the complete solution with all TODOs filled in and discussion answers provided.

## What is RAG?

Large Language Models (like GPT-4) are incredibly powerful: you can ask them questions and they generate fluent, convincing answers. But they have a **big limitation**: they can only answer from what they memorized during training. If you ask about something they haven't seen — for example, a company's internal documents, or news from last week — they might **hallucinate**: confidently make up an answer that sounds right but is completely wrong.

**Retrieval-Augmented Generation (RAG)** solves this problem by adding a **search step** before generating the answer.

### A real-world analogy

Imagine you're taking an exam:
- **Without RAG** = a **closed-book exam**: you answer from memory. If you don't remember something, you might guess (and guess wrong).
- **With RAG** = an **open-book exam**: before answering, you search your textbook for the relevant pages, read them, and *then* answer. Much more reliable!

### The RAG Pipeline — Step by Step

```
Documents → Chunk → Embed → Store in Vector DB
                                    ↓
User Query → Embed Query → Retrieve Top-k → Augment Prompt → LLM → Answer
```

| Step | What it does | Analogy |
|------|-------------|---------|
| **Chunk** | Split long documents into shorter passages (~500 chars each) | Cutting a book into individual pages |
| **Embed** | Convert each passage into a list of numbers (a vector) that captures its meaning | Creating a "fingerprint" of the text's meaning |
| **Store** | Save all vectors in a special database optimized for finding similar items | Filing the fingerprints in a cabinet |
| **Retrieve** | When a user asks a question, find the most similar passages | Looking up the most relevant pages |
| **Augment** | Insert those passages into the LLM's prompt | Handing the relevant pages to the student |
| **Answer** | The LLM reads the passages and generates an answer | The student reads and answers |

---
## Setup

| Library | What it does in our exercise |
|---------|------------------------------|
| `datasets` | Downloads the SQuAD dataset from HuggingFace (a hub for AI datasets) |
| `langchain` | A framework that helps us connect the different pieces of the RAG pipeline together, like LEGO blocks |
| `chromadb` | A vector database — think of it as a "smart filing cabinet" that can find similar documents |
| `openai` | Lets us use OpenAI's models for generating embeddings and answers |
| `helper` | Our own utility module with plotting and display functions |

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/rag_exercise')

sys.path.insert(0, '.')

!pip install datasets langchain langchain-openai langchain-community chromadb openai tiktoken matplotlib numpy --quiet
print('Setup complete!')

In [ ]:
import helper
from helper import format_docs

from datasets import load_dataset
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
print('RAG Pipeline loaded!')

In [ ]:
# ── API Key ────────────────────────────────────────────────────────────────
# os.environ["OPENAI_API_KEY"] = "sk-..."
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key
except (ImportError, Exception):
    pass

assert os.environ.get("OPENAI_API_KEY"), "Set your OPENAI_API_KEY before continuing."
print('API key set!')

---
## Task 1 — Data Loading & Exploration

### What we're doing and why

Every RAG system needs a **knowledge base** — a collection of documents that the system can search through. Think of it as the "textbook" that our system will consult during the open-book exam.

For this exercise, we use Wikipedia paragraphs from the **SQuAD dataset** (Stanford Question Answering Dataset). SQuAD is a famous dataset used in AI research: it contains thousands of question–answer pairs, where each question refers to a specific Wikipedia paragraph.

**Example from SQuAD:**
> **Paragraph**: "The Normans were the people who gave their name to Normandy, a region in France..."
> **Question**: "In what country is Normandy located?"
> **Answer**: "France"

Since multiple questions can refer to the same paragraph, we first **deduplicate** (remove duplicates from) the paragraphs to get our unique list of documents.

### Nothing to code here — just run and observe!

In [ ]:
# ── Load the SQuAD v2 dataset from HuggingFace ───────────────────────────
squad = load_dataset("rajpurkar/squad_v2", split="validation")

print(f"Total examples in validation set: {len(squad)}")
print(f"Each example contains: {list(squad.features.keys())}")
print(f"We care about 'context' (the Wikipedia paragraph) and 'question'/'answers'.")

In [ ]:
# ── Extract unique documents ──────────────────────────────────────────────
all_documents = list(dict.fromkeys(squad["context"]))
documents = all_documents[:200]

print(f"Unique documents: {len(all_documents)} | Working set: {len(documents)}")
print(f"\nFirst document (first 300 chars):")
print(f"{documents[0][:300]}...")

In [ ]:
# ── Visualize document length distribution ───────────────────────────────
helper.plot_document_length_distribution(documents)

### Discussion — SOLUTION

The documents vary significantly in length, ranging from a few hundred to over a thousand characters. Most documents fall in the 400–800 character range. This is typical of Wikipedia paragraphs: some topics are covered briefly, others in more detail.

**Why this matters for chunking:** Documents longer than our chunk size (500 characters) will be split into multiple chunks, while shorter documents will remain as a single chunk. The uneven distribution means some topics will have more chunks (and thus more chances to be retrieved) than others.

**The problem:** If we want to feed these to an embedding model that works best with ~500 characters, the longer documents are too big and need to be split up — which is exactly what we do in Task 2.

---
## Task 2 — Document Chunking

### Why do we need to cut documents into pieces?

Imagine you have a 10-page article about World War II. If someone asks "When did D-Day happen?", the answer is in just one paragraph. But if we store the entire article as a single unit, our search system would return *all 10 pages* — and the LLM would have to read through everything to find the one relevant paragraph.

**Chunking** = splitting long documents into shorter, focused passages. This helps in two ways:

1. **Better search precision**: a short chunk about "D-Day, June 6, 1944" will match the question much better than a long article that mentions D-Day once among many other topics.
2. **Fits model limits**: embedding models have a maximum input length. Shorter chunks ensure we stay within limits.

### How does `RecursiveCharacterTextSplitter` work?

LangChain provides a smart splitter that tries to split at **natural boundaries**, in this priority order:

1. **Double newlines** (paragraph breaks) — best: keeps whole paragraphs together
2. **Single newlines** — good: keeps sentences together
3. **Spaces** (between words) — okay: at least doesn't cut words in half
4. **Individual characters** — last resort

We configure it with two numbers:
- **`chunk_size=500`** — each chunk should be at most ~500 characters long
- **`chunk_overlap=50`** — the last 50 characters of one chunk are repeated at the start of the next chunk

**Why overlap?** Imagine a sentence that says "Napoleon was born in **Corsica** in 1769". If the chunk boundary falls right between "Corsica" and "in 1769", one chunk would have the place without the date, and the other would have the date without the place. Overlap ensures that boundary information isn't lost.

---

### 🎯 SOLUTION — What students needed to do

Students had to replace `___` with the correct values:
1. `chunk_size=500` and `chunk_overlap=50`
2. Pass `documents` to `create_documents()`

In [ ]:
# ── Task 2: Create the text splitter and split documents ─────────────────

# 🎯 SOLUTION: the blanks were 500 and 50
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# 🎯 SOLUTION: the blank was "documents"
chunks = text_splitter.create_documents(documents)

print(f"Split {len(documents)} documents → {len(chunks)} chunks "
      f"(avg {len(chunks)/len(documents):.1f} per doc)")
print(f"\nFirst chunk:\n{chunks[0].page_content[:200]}...")

In [ ]:
# ── Visualize chunk length distribution ──────────────────────────────────
helper.plot_chunk_length_distribution(chunks)

### Discussion — SOLUTION

1. **Very small chunks** (e.g., 50 characters): Each chunk would be just a sentence fragment like "in France during the". The embedding of such a short fragment doesn't carry enough meaning to match well against a question. Retrieval would be unreliable because the search system can't understand what such a tiny piece of text is about.

2. **Very large chunks** (e.g., 5000 characters): Each chunk would contain multiple topics. The embedding would "average" across all of them, making it a poor match for any specific question. Also, we'd send a lot of irrelevant text to the LLM, wasting tokens and potentially confusing it.

3. **Chunk overlap (50 characters)**: Without overlap, a sentence like "Napoleon was born in Corsica in 1769" might get split right in the middle — one chunk would have "Napoleon was born in Corsica" and the next would start with "in 1769". Neither chunk alone contains the full information. The overlap ensures that boundary sentences appear in both chunks, so important information isn't lost.

---
## Task 3 — Embeddings & Vector Store

### What are embeddings? (The core idea!)

This is one of the most important concepts in modern AI. An **embedding** is a way to represent text as a **list of numbers** (called a "vector"). The magic is that **texts with similar meanings get similar numbers**.

**Example:**
- "The cat sat on the mat" → `[0.12, -0.45, 0.78, ...]` (1536 numbers)
- "A kitten rested on the rug" → `[0.11, -0.43, 0.80, ...]` (very similar numbers!)
- "The stock market crashed" → `[0.89, 0.23, -0.56, ...]` (very different numbers!)

Even though "cat" and "kitten" are different words, the embedding model understands they mean similar things. This is what makes semantic search possible: instead of matching exact words (like Ctrl+F), we match **meanings**.

### What is a vector store?

A **vector store** (we use ChromaDB) is a database specifically designed for storing and searching these number-lists efficiently. It's like a library catalog, but instead of searching by title or author, you search by *meaning*.

### What happens step by step when we run `Chroma.from_documents()`

1. Each chunk's text is sent to OpenAI's embedding model (via the API)
2. The model converts each chunk into a list of 1536 numbers
3. ChromaDB stores both the original text AND its numbers
4. Later, when you search, your question is also converted to numbers, and ChromaDB finds the closest match

---

### 🎯 SOLUTION — What students needed to do

Students had to replace `___` with the correct values:
1. The model name `"text-embedding-3-small"`
2. The two arguments to `Chroma.from_documents()`: `chunks` and `embeddings`

In [ ]:
# ── Task 3: Create embedding model and vector store ──────────────────────

# 🎯 SOLUTION: the blank was "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 🎯 SOLUTION: the blanks were chunks, embeddings
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready — {vectorstore._collection.count()} vectors indexed.")

In [ ]:
# ── Test retrieval ────────────────────────────────────────────────────────
test_queries = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain!
]

for query in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    helper.display_retrieval_results(query, results)

### What just happened?

The vector store **always returns something**, even for the pizza question — it returned the "least unrelated" chunks. This is a fundamental property of vector search: it can't say "I have nothing relevant." Notice the higher distance scores for the pizza query.

### Discussion — SOLUTION

1. **In-domain queries**: The retrieved passages are relevant and contain information that could answer the questions. The distance scores are low, indicating strong semantic matches.

2. **Out-of-domain query (pizza)**: The vector store still returns results! It finds the "least dissimilar" chunks, but they're clearly unrelated to pizza. The distance scores are noticeably higher.

3. **Score comparison**: Yes, the pizza query scores are higher (= less similar) than the in-domain queries.

4. **Key danger**: Since the vector store ALWAYS returns something (it can't say "I have nothing relevant"), if we feed these irrelevant chunks to an LLM without proper instructions, the LLM might try to fabricate an answer about pizza from completely unrelated Wikipedia text. This is why our prompt template MUST instruct the LLM to say "I don't know" when the context isn't helpful.

---
## Task 4 — Build the RAG Chain

### Connecting all the pieces

Now we connect all the components into a working pipeline. Think of it like an assembly line in a factory:

```
User's question
      ↓
[1. RETRIEVER] → searches the vector store → finds relevant chunks
      ↓
[2. FORMAT]    → joins the chunks into a single text block called "context"
      ↓
[3. PROMPT]    → inserts the context + question into a template for the LLM
      ↓
[4. LLM]       → reads the prompt and generates an answer
      ↓
[5. PARSER]    → extracts just the text from the LLM's response
      ↓
Final answer!
```

### What is a prompt template?

The **prompt template** is the set of instructions we give to the LLM. It's like briefing a human assistant before asking them a question. It contains two special **placeholders** that LangChain fills in automatically:
- **`{context}`** → gets replaced with the retrieved chunks
- **`{question}`** → gets replaced with the user's question

A good RAG prompt must tell the LLM two critical rules:
1. **Answer ONLY from the context** — otherwise the LLM ignores the retrieved documents and answers from memory (defeating the purpose of RAG!)
2. **Say "I don't know" if the context doesn't help** — remember the pizza question? The vector store always returns *something*, even for irrelevant queries. Without this rule, the LLM would make up answers.

---

### 🎯 SOLUTION — What students needed to do

Students had to write 2–3 sentences of instructions in plain English replacing `___WRITE YOUR INSTRUCTIONS HERE___`. The key requirements were:
- Tell the LLM to answer ONLY based on the context
- Tell the LLM to say "I don't know" if the context doesn't help
- Keep answers concise

In [ ]:
# ── Create retriever and LLM ──────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print('Retriever (k=4) and LLM (gpt-4o-mini) ready!')

In [ ]:
# ── Task 4: Write the prompt template ────────────────────────────────────
#
# 🎯 SOLUTION: Students had to write their own instructions here.
#    This is one possible version — any prompt that includes the three key rules works.

RAG_PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

Rules:
- Answer ONLY based on the context below. Do not use any prior knowledge.
- If the context does not contain enough information to answer the question, say "I don't know".
- Keep your answer concise and to the point.

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

In [ ]:
# ── Build the RAG chain ───────────────────────────────────────────────────
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print('RAG chain ready!')

In [ ]:
# ── Test the RAG chain ────────────────────────────────────────────────────
test_questions = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain — will it say "I don't know"?
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}\nA: {answer}\n")

### Discussion — SOLUTION

The RAG chain correctly answers in-domain questions by synthesizing information from the retrieved Wikipedia passages. For the out-of-domain pizza question, it says "I don't know" because the prompt explicitly instructs it to only answer from the context.

**Without the "I don't know" instruction**, the LLM would likely answer the pizza question from its own memory (e.g., "Many people enjoy mozzarella..."), which defeats the whole purpose of RAG.

**Without the "only use context" instruction**, the LLM might mix its own knowledge with the retrieved chunks, making it impossible to know whether the answer came from our knowledge base or from hallucination.

The prompt is our main tool for controlling the LLM's behavior — even small changes to the wording can significantly affect the output. This is why "prompt engineering" is such an important skill in AI.

---
## Task 5 — Evaluation & Comparison

### Why do we need a proper evaluation?

Testing with 3 questions gave us a first impression, but it's not enough to know if our pipeline works *well*. We need to:
1. Test on **many** questions (we'll use 10)
2. Have **ground-truth answers** to compare against
3. Use a **metric** (a number) to measure performance

### Our metric: substring accuracy

> For each question, check if the expected answer appears *somewhere* inside the generated answer (ignoring upper/lower case).

| Gold answer | Generated answer | Correct? | Why? |
|-------------|-----------------|----------|------|
| "France" | "The answer is France, a country in Europe" | ✓ Yes | "france" is inside the text |
| "France" | "Germany" | ✗ No | "france" is NOT inside the text |
| "1066" | "The Battle of Hastings took place in 1066" | ✓ Yes | "1066" is inside the text |

### RAG vs. Naive: does retrieval actually help?

We compare:
- **RAG chain**: retrieves relevant chunks, then answers (the "open-book exam")
- **Naive chain**: sends only the question to the LLM, with NO context (the "closed-book exam")

---

### 🎯 SOLUTION — What students needed to do

1. **Task 5a**: Connect three components with `|`: `naive_prompt | llm | StrOutputParser()`
2. **Task 5b**: Replace `___` with `rag_answers` and `naive_answers` in the accuracy formulas

In [ ]:
# ── Build evaluation set from SQuAD ──────────────────────────────────────
eval_set = helper.build_eval_set(squad, documents, n=30)
print(f"Evaluation set: {len(eval_set)} questions")
print(f"\nExamples:")
for i, ex in enumerate(eval_set[:3]):
    print(f"  [{i+1}] Q: {ex['question']}  →  A: {ex['answer']}")

In [ ]:
# ── Run RAG chain on evaluation questions ────────────────────────────────
eval_questions = [ex["question"] for ex in eval_set[:10]]
gold_answers = [ex["answer"] for ex in eval_set[:10]]

print("Running RAG chain on 10 evaluation questions...\n")
rag_answers = []
for i, q in enumerate(eval_questions):
    answer = rag_chain.invoke(q)
    rag_answers.append(answer)
    correct = gold_answers[i].lower() in answer.lower()
    mark = "✓" if correct else "✗"
    print(f"  [{i+1}/10] {mark}  Q: {q}")
    print(f"           A: {answer}  |  Gold: {gold_answers[i]}\n")

In [ ]:
# ── Task 5a: Build the naive chain (the "closed-book" version) ───────────
naive_prompt = ChatPromptTemplate.from_template(
    "Answer the following question concisely:\n\n{question}"
)

# 🎯 SOLUTION: students had to write: naive_prompt | llm | StrOutputParser()
naive_chain = naive_prompt | llm | StrOutputParser()
print('Naive chain ready!')

In [ ]:
# ── Run naive chain on the same questions ────────────────────────────────
print("Running naive chain (NO retrieval) on the same 10 questions...\n")
naive_answers = []
for i, q in enumerate(eval_questions):
    answer = naive_chain.invoke(q)
    naive_answers.append(answer)
    correct = gold_answers[i].lower() in answer.lower()
    mark = "✓" if correct else "✗"
    print(f"  [{i+1}/10] {mark}  Q: {q}")
    print(f"           A: {answer}\n")

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────
helper.display_comparison_table(eval_questions, rag_answers, naive_answers, gold_answers)

In [ ]:
# ── Task 5b: Compute substring accuracy ──────────────────────────────────

# 🎯 SOLUTION: the blank was "rag_answers"
rag_accuracy = sum(
    1 for gen, gold in zip(rag_answers, gold_answers)
    if gold.lower() in gen.lower()
) / len(eval_questions)

# 🎯 SOLUTION: the blank was "naive_answers"
naive_accuracy = sum(
    1 for gen, gold in zip(naive_answers, gold_answers)
    if gold.lower() in gen.lower()
) / len(eval_questions)

print(f"RAG accuracy:   {rag_accuracy:.0%} ({int(rag_accuracy * len(eval_questions))}/{len(eval_questions)})")
print(f"Naive accuracy: {naive_accuracy:.0%} ({int(naive_accuracy * len(eval_questions))}/{len(eval_questions)})")

In [ ]:
# ── Visualize the comparison ──────────────────────────────────────────────
helper.plot_evaluation_results({
    "RAG Accuracy": rag_accuracy,
    "Naive Accuracy": naive_accuracy,
})

### Final Discussion — SOLUTION

1. **Which chain won?** RAG generally outperforms the naive LLM, especially on questions about **specific factual details** (dates, names, numbers) that appear in the Wikipedia passages.

2. **When naive wins:** The naive LLM can answer **well-known general knowledge questions** correctly because they were heavily represented in its training data. For these, RAG doesn't add much.

3. **When RAG loses:** If the retriever returns **irrelevant chunks**, the LLM might get confused or say "I don't know" even though it could have answered from memory. Also, if the **gold answer phrasing** doesn't match the LLM's phrasing (e.g., gold: "1066", LLM: "the year 1066 AD"), substring matching gives a false negative.

4. **Real-world applications:** RAG is essential when the system needs to answer from **private or recent data** that the LLM has never seen during training — company documents, legal contracts, updated product manuals, medical records, etc.

5. **Biggest impact:** The prompt template and the chunking strategy typically have the biggest impact. A bad prompt leads to hallucination; bad chunking leads to irrelevant retrieval.